In [ ]:
import copy
import math
import torch
import numpy as np
import torch.nn as nn
from torch import Tensor
from functools import partial
import torchvision.transforms.functional as F
import torch.nn.functional as F_nn
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple, Union
from torchvision.models.resnet import BasicBlock, Bottleneck, conv1x1
from torchvision.transforms import InterpolationMode
import ee
import pandas as pd
import openpyxl
import datetime
import os
from PIL import Image
import numpy as np
import json
import IPython.display as disp
import os
import json
import torch
import numpy as np
from PIL import Image
from skimage.io import imread
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader, SubsetRandomSampler
from torchvision import transforms
from skimage.io import imread
import matplotlib.pyplot as plt

In [ ]:
!pip install --upgrade pillow

In [ ]:
!cp "/content/drive/Othercomputers/MiPC/Code_OilSpill/training/encoder_models.py" "/content/"
!cp "/content/drive/Othercomputers/MiPC/Code_OilSpill/training/decoder_models.py" "/content/"
!cp "/content/drive/Othercomputers/MiPC/Code_OilSpill/training/seg_models.py" "/content/"

In [ ]:
# Set the paths to your dataset
#dir_data = '/content/drive/MyDrive/CIMA2023/Documentos2023/Proyectos/Proy1-ImagenesEspectrales/data/imagenes/OilDataset/train/'
list_images = os.listdir(os.path.join(dir_data, "images"))
#sos_dir = '/content/drive/Othercomputers/MiPC/filtered_patches/train'
krest_dir = '/content/drive/Othercomputers/MiPC/subset_TRAIN'


In [ ]:
krest_dir = '/content/drive/Othercomputers/MiPC/subset_TRAIN'

In [ ]:
from torchvision.models.resnet import (
    ResNet18_Weights,
    ResNet34_Weights,
    ResNet50_Weights,
    ResNet101_Weights,
)
import torch

In [ ]:
class M4DSAROilSpillDataset(Dataset):
    def __init__(
        self,
        images_dir,
        labels_dir,
        transform=None,
        augmentation=None,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.augmentation = augmentation

        # Get list of image files
        self.image_files = sorted([
            f for f in os.listdir(self.images_dir)
            if f.endswith('.jpg') or f.endswith('.png')
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Get image file path
        image_file = self.image_files[idx]
        image_path = os.path.join(self.images_dir, image_file)

        # Load image
        image = Image.open(image_path).convert('RGB')

        # Construct corresponding label file name
        if '_sat.jpg' in image_file:
            # SOS Dataset
            label_file = image_file.replace('_sat.jpg', '_label_1D.png')
        elif image_file.startswith('img_'):
            # Krestininis Dataset
            label_file = image_file.replace('.png', '_label_1D.png')
        else:
            raise ValueError(f'Unknown image file format: {image_file}')

        label_path = os.path.join(self.labels_dir, label_file)
        if not os.path.exists(label_path):
            raise FileNotFoundError(f"Label file not found: {label_path}")

        label = Image.open(label_path).convert('L')

        # Apply augmentation if any
        if self.augmentation:
            image, label = self.augmentation(image, label)

        # Apply transformations
        if self.transform:
            image = self.transform(image)
            label = torch.from_numpy(np.array(label, dtype=np.int64))

        return image, label


CAMBIAR NORMALIZACION VALUES

In [ ]:
import torchvision.transforms as transforms
import random

# Image normalization values (adjust based on your dataset statistics)
mean = [0.4853970078639312, 0.4853970078639312, 0.4853970078639312]  # Example values; adjust if necessary
std = [0.17816201511895402, 0.17816201511895402, 0.17816201511895402]

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

def augmentation(image, label):
    # Random horizontal flip
    if random.random() > 0.5:
        image = F.hflip(image)
        label = F.hflip(label)
    # Random vertical flip
    if random.random() > 0.5:
        image = F.vflip(image)
        label = F.vflip(label)
    # Random rotation
    angle = random.choice([0, 90, 180, 270])
    if angle != 0:
        image = F.rotate(image, angle, interpolation=InterpolationMode.BILINEAR)
        label = F.rotate(label, angle, interpolation=InterpolationMode.NEAREST)
    return image, label


In [ ]:
from torch.utils.data import DataLoader, ConcatDataset, SubsetRandomSampler
import numpy as np

def get_dataloaders_for_training(
    sos_dir,
    krest_dir,
    batch_size,
    random_state=None,
    num_workers=4,
):
    datasets = []

    # Handle SOS dataset
    if sos_dir:
        sos_train_images_dir = os.path.join(sos_dir, 'train')
        sos_train_labels_dir = os.path.join(sos_dir, 'train_1D')
        sos_train_dataset = M4DSAROilSpillDataset(
            images_dir=sos_train_images_dir,
            labels_dir=sos_train_labels_dir,
            transform=image_transform,
            augmentation=augmentation,
        )
        datasets.append(sos_train_dataset)

    # Handle Krestininis dataset
    if krest_dir:
        #krest_train_images_dir = os.path.join(krest_dir, 'train', 'images')
        #krest_train_labels_dir = os.path.join(krest_dir, 'train', 'labels_1D')
        krest_train_images_dir = os.path.join(krest_dir, 'images')
        krest_train_labels_dir = os.path.join(krest_dir, 'labels_1D')
        krest_train_dataset = M4DSAROilSpillDataset(
            images_dir=krest_train_images_dir,
            labels_dir=krest_train_labels_dir,
            transform=image_transform,
            augmentation=augmentation,
        )
        datasets.append(krest_train_dataset)

    # Combine datasets
    if not datasets:
        raise ValueError("No datasets provided for training.")
    combined_train_dataset = ConcatDataset(datasets)

    # Create indices for splitting dataset into training and validation
    dataset_size = len(combined_train_dataset)
    indices = list(range(dataset_size))
    split = int(np.floor(0.05 * dataset_size))  # 5% for validation
    if random_state is not None:
        np.random.seed(random_state)
    np.random.shuffle(indices)
    train_indices, valid_indices = indices[split:], indices[:split]

    # Create samplers
    train_sampler = SubsetRandomSampler(train_indices)
    valid_sampler = SubsetRandomSampler(valid_indices)

    # Create data loaders
    train_loader = DataLoader(
        combined_train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        num_workers=num_workers,
    )

    valid_loader = DataLoader(
        combined_train_dataset,
        batch_size=batch_size,
        sampler=valid_sampler,
        num_workers=num_workers,
    )

    print("Dataset information:")
    print(f"Number of training samples: {len(train_indices)}")
    print(f"Number of validation samples: {len(valid_indices)}")

    return train_loader, valid_loader


In [ ]:
def get_dataloader_for_inference(
    sos_dir=None,
    krest_dir=None,
    batch_size=1,
    num_workers=4,
):
    datasets = []

    # Handle SOS dataset
    if sos_dir:
        sos_test_images_dir = os.path.join(sos_dir, 'test')
        sos_test_labels_dir = os.path.join(sos_dir, 'test_1D')
        sos_test_dataset = M4DSAROilSpillDataset(
            images_dir=sos_test_images_dir,
            labels_dir=sos_test_labels_dir,
            transform=image_transform,
        )
        datasets.append(sos_test_dataset)

    # Handle Krestininis dataset
    if krest_dir:
        krest_test_images_dir = os.path.join(krest_dir, 'test', 'images')
        krest_test_labels_dir = os.path.join(krest_dir, 'test', 'labels_1D')
        krest_test_dataset = M4DSAROilSpillDataset(
            images_dir=krest_test_images_dir,
            labels_dir=krest_test_labels_dir,
            transform=image_transform,
        )
        datasets.append(krest_test_dataset)

    # Combine datasets
    if not datasets:
        raise ValueError("No datasets provided for inference.")
    combined_test_dataset = torch.utils.data.ConcatDataset(datasets)

    # Create data loader
    test_loader = DataLoader(
        combined_test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return test_loader, combined_test_dataset


In [ ]:
class CustomResNet(nn.Module):
    def __init__(
        self,
        layers: List[int],
        block=BasicBlock,
        zero_init_residual=False,
        groups=1,
        num_classes=1000,
        width_per_group=64,
        replace_stride_with_dilation=None,
        norm_layer=None,
    ):
        """
        CustomResNet class to build the CustomResNet encoder model

        ----------
        Attributes
        ----------
        layers : list
            list of number of layers in each residual block
        block : object of block type
            type of the residual block (options = [BasicBlock, Bottleneck])
        zero_init_residual : bool
            to indicate whether to use zero weights for BN
        groups : int
            indicates the number of groups (default: 1)
        num_classes : int
            indicates the number of classes (default: 1000)
        width_per_group : int
            indicates the width per group (default: 64)
        replace_stride_with_dilation : list
            a list indicating whether to replace stride with dilation (default: None)
        norm_layer : object
            object of type batch norm (default: None)
        """

        super(CustomResNet, self).__init__()

        self.dict_encoder_features = {}

        if norm_layer is None:
            self._norm_layer = nn.BatchNorm2d

        self.inplanes = 64
        self.dilation = 1

        if replace_stride_with_dilation is None:
            # each element in the tuple indicates if we should replace
            # the 2x2 stride with a dilated convolution instead
            replace_stride_with_dilation = [False, False, False]

        if len(replace_stride_with_dilation) != 3:
            raise ValueError(
                "replace_stride_with_dilation should be None "
                f"or a 3-element tuple, got {replace_stride_with_dilation}"
            )

        self.groups = groups
        self.base_width = width_per_group

        self.conv1 = nn.Conv2d(
            3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.bn1 = self._norm_layer(self.inplanes) #batch normalization layer
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(
            block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0]
        )
        self.layer3 = self._make_layer(
            block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1]
        )
        self.layer4 = self._make_layer(
            block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2]
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)  # type: ignore[arg-type]

    def _make_layer(
        self,
        block,
        planes,
        blocks,
        stride=1,
        dilate=False,
    ):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes,
                planes,
                stride,
                downsample,
                self.groups,
                self.base_width,
                previous_dilation,
                norm_layer,
            )
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes,
                    planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    dilation=self.dilation,
                    norm_layer=norm_layer,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        """
        ---------
        Arguments
        ---------
        x : torch tensor
            a tensor of input features

        -------
        Returns
        -------
        x : torch tensor
            output of the CustomResNet
        """
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        self.dict_encoder_features["block_1"] = x

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        return x


def _resnet(block_type, layers, weights=None, progress=True):
    """
    ---------
    Arguments
    ---------
    block_type : object
        object of type block
    layers : list
        list of layers in each residual block
    weights : object
        object of type ResNet weights
    progress : bool
        indicates whether to show progress or not

    -------
    Returns
    -------
    model : object
        model object of type CustomResNet
    """
    model = CustomResNet(layers, block_type)

    if weights is not None:
        model.load_state_dict(weights.get_state_dict(progress=progress))

    return model

In [ ]:
def compute_mean_pixel_acc(true_label, pred_label):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label

    -------
    Returns
    -------
    mean_pixel_accuracy : float
        mean pixel accuracy
    """
    if true_label.shape != pred_label.shape:
        print(
            "true_label has dimension",
            true_label.shape,
            ", pred_label values have shape",
            pred_label.shape,
        )
        return

    if true_label.dim() != 3:
        print("true_label has dim", true_label.dim(), ", Must be 3.")
        return

    acc_sum = 0
    for i in range(true_label.shape[0]):
        true_label_arr = true_label[i, :, :].clone().detach().cpu().numpy()
        pred_label_arr = pred_label[i, :, :].clone().detach().cpu().numpy()
        true_label_arr = true_label_arr.astype(np.int32)
        pred_label_arr = pred_label_arr.astype(np.int32)

        same = (true_label_arr == pred_label_arr).sum()

        a, b = true_label_arr.shape
        total = a * b

        acc_sum += same / total

    mean_pixel_accuracy = acc_sum / true_label.shape[0]
    return mean_pixel_accuracy


# compute mean IOU
def compute_mean_IOU(true_label, pred_label, num_classes=5):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label
    num_classes : int
        number of classes in the dataset (default: 5)

    -------
    Returns
    -------
    mean_iou : float
        mean IoU
    """
    iou_list = list()
    present_iou_list = list()

    pred_label = pred_label.view(-1)
    true_label = true_label.view(-1)
    # Note: Following for loop goes from 0 to (num_classes-1)
    # in computation of IoU.
    for sem_class in range(num_classes):
        pred_label_inds = pred_label == sem_class
        target_inds = true_label == sem_class
        if target_inds.long().sum().item() == 0:
            iou_now = float("nan")
        else:
            intersection_now = (pred_label_inds[target_inds]).long().sum().item()
            union_now = (
                pred_label_inds.long().sum().item()
                + target_inds.long().sum().item()
                - intersection_now
            )
            iou_now = float(intersection_now) / float(union_now)
            present_iou_list.append(iou_now)
        iou_list.append(iou_now)
    present_iou_list = np.array(present_iou_list)
    return np.mean(present_iou_list)

def compute_mean_IOU(true_label, pred_label, num_classes=5):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label
    num_classes : int
        number of classes in the dataset (default: 5)

    -------
    Returns
    -------
    mean_iou : float
        mean IoU
    """
    iou_list = list()
    present_iou_list = list()

    pred_label = pred_label.view(-1)
    true_label = true_label.view(-1)
    # Note: Following for loop goes from 0 to (num_classes-1)
    # in computation of IoU.
    for sem_class in range(num_classes):
        pred_label_inds = pred_label == sem_class
        target_inds = true_label == sem_class
        if target_inds.long().sum().item() == 0:
            iou_now = float("nan")
        else:
            intersection_now = (pred_label_inds[target_inds]).long().sum().item()
            union_now = (
                pred_label_inds.long().sum().item()
                + target_inds.long().sum().item()
                - intersection_now
            )
            iou_now = float(intersection_now) / float(union_now)
            present_iou_list.append(iou_now)
        iou_list.append(iou_now)
    present_iou_list = np.array(present_iou_list)
    return np.mean(present_iou_list)


def compute_class_IOU(true_label, pred_label, num_classes=5):
    """
    ---------
    Arguments
    ---------
    true_label : ndarray
        a numpy array of groundtruth label
    pred_label : ndarray
        a numpy array of prediction label
    num_classes : int
        number of classes in the dataset (default: 5)


    -------
    Returns
    -------
    per_class_iou : ndarray
        a numpy array of per class IoU
    """
    iou_list = list()
    present_iou_list = list()

    pred_label = pred_label.view(-1)
    true_label = true_label.view(-1)

    per_class_iou = np.zeros(num_classes)

    # Note: Following for loop goes from 0 to (num_classes-1)
    # in computation of IoU.
    for sem_class in range(num_classes):
        pred_label_inds = pred_label == sem_class
        target_inds = true_label == sem_class
        if target_inds.long().sum().item() == 0:
            iou_now = float("nan")
        else:
            intersection_now = (pred_label_inds[target_inds]).long().sum().item()
            union_now = (
                pred_label_inds.long().sum().item()
                + target_inds.long().sum().item()
                - intersection_now
            )
            iou_now = float(intersection_now) / float(union_now)
            present_iou_list.append(iou_now)
        per_class_iou[sem_class] = iou_now
    return per_class_iou


In [ ]:
from torch.optim.lr_scheduler import _LRScheduler
def validation_loop(dataset_loader, model, ce_loss, device):
    """
    ---------
    Arguments
    ---------
    dataset_loader : object
        object of type dataloader
    model : object
        object of type model
    ce_loss : object
        object of type cross entropy loss
    device : str
        device on which training needs to be run

    -------
    Returns
    -------
    (valid_loss, valid_acc, valid_IOU) : tuple
        a tuples of torch floats of mean loss, mean accuracy, mean IoU for the validation set
    """
    model.eval()
    size = len(dataset_loader.dataset)
    num_batches = len(dataset_loader)
    valid_loss, valid_acc, valid_IOU = 0, 0, 0

    with torch.no_grad():
        for image, label in dataset_loader:
            image = image.to(device, dtype=torch.float)
            label = label.to(device, dtype=torch.long)

            pred_logits = model(image)
            valid_loss += ce_loss(pred_logits, label)

            pred_probs = F_nn.softmax(pred_logits, dim=1)
            pred_label = torch.argmax(pred_probs, dim=1)

            valid_acc += compute_mean_pixel_acc(label, pred_label)
            valid_IOU += compute_mean_IOU(label, pred_label)

    valid_loss /= num_batches
    valid_acc /= num_batches
    valid_IOU /= num_batches
    return valid_loss, valid_acc, valid_IOU


def train_loop(dataset_loader, model, ce_loss, optimizer, device):
    """
    ---------
    Arguments
    ---------
    dataset_loader : object
        object of type dataloader
    model : object
        object of type model
    ce_loss : object
        object of type cross entropy loss
    optimizer : object
        object of type optimizer
    device : str
        device on which training needs to be run

    -------
    Returns
    -------
    train_loss : torch float
        mean loss for the training set
    """
    model.train()
    size = len(dataset_loader.dataset)
    num_batches = len(dataset_loader)
    train_loss = 0

    for image, label in dataset_loader:
        image = image.to(device, dtype=torch.float)
        label = label.to(device, dtype=torch.long)
        optimizer.zero_grad()

        pred_logits = model(image)
        loss = ce_loss(pred_logits, label)

        # Backpropagation
        loss.backward()
        optimizer.step()

        train_loss += loss
    train_loss /= num_batches
    return train_loss

class PolynomialLR(_LRScheduler):
    """
    PolynomialLR class for the polynomial learning rate scheduler

    ----------
    Attributes
    ----------
    optimizer : object
        object of type optimizer
    max_epochs : int
        maximum number of epochs for which optimization needs to be run
    power : float
        the power term in the polynomial learning rate scheduler (default: 0.9)
    last_epoch : int
        last epoch in the optimization (default: -1)
    min_lr : float
        minimum value for the learning rate (default: 1e-6)
    """

    def __init__(self, optimizer, max_epochs, power=0.9, last_epoch=-1, min_lr=1e-6):
        self.power = power
        self.max_epochs = max_epochs
        self.min_lr = min_lr  # avoid zero lr
        super(PolynomialLR, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        return [
            max(
                base_lr * (1 - self.last_epoch / self.max_epochs) ** self.power,
                self.min_lr,
            )
            for base_lr in self.base_lrs
        ]


In [ ]:
!cp "/content/drive/Othercomputers/MiPC/Code_OilSpill/training/logger_utils.py" "/content/"
from logger_utils import CSVWriter, write_dict_to_json
from seg_models import ResNet34DeepLabV3Plus

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.epochs_no_improve = 0
        self.early_stop = False

    def __call__(self, validation_loss):
        if self.best_score is None:
            self.best_score = validation_loss
        elif validation_loss < self.best_score - self.min_delta:
            self.best_score = validation_loss
            self.epochs_no_improve = 0
        else:
            self.epochs_no_improve += 1
            if self.epochs_no_improve >= self.patience:
                self.early_stop = True
early_stopping = EarlyStopping(patience=5, min_delta=0.001)

In [ ]:
import time
import os
def batch_train(FLAGS):
    dir_path = os.path.join(FLAGS.dir_model, FLAGS.which_model)
    if not os.path.isdir(dir_path):
        os.makedirs(dir_path)
        print(f"Created directory: {dir_path}")
    csv_writer = CSVWriter(
        file_name=os.path.join(dir_path, "train_metrics.csv"),
        column_names=["epoch", "train_loss", "valid_loss", "valid_acc", "valid_IOU"],
    )

    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Get data loaders
    sos_dir = FLAGS.sos_dir  # Make sure to set this in your FLAGS
    krest_dir = FLAGS.krest_dir  # Make sure to set this in your FLAGS

    train_dataset_loader, valid_dataset_loader = get_dataloaders_for_training(
        sos_dir=sos_dir,
        krest_dir=krest_dir,
        batch_size=FLAGS.batch_size,
        random_state=FLAGS.random_state,
    )

    # Model selection
    if FLAGS.which_model == "resnet_34_deeplab_v3+":
        oil_spill_seg_model = ResNet34DeepLabV3Plus(
            num_classes=FLAGS.num_classes, pretrained=bool(FLAGS.pretrained)
        )
    else:
        print("Model not implemented.")
        return

    oil_spill_seg_model.to(device)

    # Optimizer selection
    if FLAGS.which_optimizer == "sgd":
        optimizer = torch.optim.SGD(
            oil_spill_seg_model.parameters(),
            lr=FLAGS.learning_rate,
            momentum=0.9,
            weight_decay=FLAGS.weight_decay,
        )
        lr_scheduler = PolynomialLR(
            optimizer,
            FLAGS.num_epochs + 1,
            power=0.9,
        )
    elif FLAGS.which_optimizer == "adamw":
        optimizer = torch.optim.AdamW(
            oil_spill_seg_model.parameters(),
            lr=FLAGS.learning_rate,
            weight_decay=FLAGS.weight_decay,
        )
        lr_scheduler = None

    ce_loss = torch.nn.CrossEntropyLoss()
    print(f"\nTraining oil spill segmentation model: {FLAGS.which_model}\n")

    # Save training parameters
    serializable_flags = {k: v for k, v in vars(FLAGS).items() if isinstance(v, (int, float, str, list, dict))}
    write_dict_to_json(os.path.join(dir_path, "params.json"), serializable_flags)

    for epoch in range(1, FLAGS.num_epochs + 1):
        t_1 = time.time()
        train_loss = train_loop(
            train_dataset_loader, oil_spill_seg_model, ce_loss, optimizer, device
        )
        t_2 = time.time()
        print("-" * 100)
        print(
            f"Epoch : {epoch}/{FLAGS.num_epochs}, Time: {(t_2 - t_1):.2f} sec., Train Loss: {train_loss:.5f}"
        )
        valid_loss, valid_acc, valid_IOU = validation_loop(
            valid_dataset_loader, oil_spill_seg_model, ce_loss, device
        )
        print(
            f"Validation Loss: {valid_loss:.5f}, Validation Accuracy: {valid_acc:.5f}, Validation IOU: {valid_IOU:.5f}"
        )

        # Early stopping (if implemented)
        # early_stopping(valid_loss)
        early_stopping(valid_loss)
        if early_stopping.early_stop:
          print("Early stopping")
          break

        csv_writer.write_row(
            [
                epoch,
                round(train_loss.item(), 5),  # Convert tensor to Python number before rounding
                round(valid_loss.item(), 5),  # Convert tensor to Python number before rounding
                round(valid_acc.item(), 5),  # Convert tensor to Python number before rounding
                round(valid_IOU.item(), 5),
            ]
        )
        torch.save(
            oil_spill_seg_model.state_dict(),
            os.path.join(dir_path, f"oil_spill_seg_{FLAGS.which_model}_{epoch}.pt"),
        )
        if lr_scheduler is not None:
            lr_scheduler.step()
    print("Training complete!")
    csv_writer.close()
    return


In [ ]:
class FLAGS:
    sos_dir = None  # Set to None if not using SOS dataset
    krest_dir = '/content/drive/Othercomputers/MiPC/subset_TRAIN'  # Ensure this path is correct
    pretrained = 1  # 1 for True, 0 for False
    random_state = 42
    which_optimizer = "sgd"  # or "adamw"
    learning_rate = 1e-2
    weight_decay = 1e-4
    num_epochs = 50
    batch_size = 4
    num_classes = 5  # Update if necessary
    which_model = "resnet_34_deeplab_v3+"
    dir_model = os.getcwd()
    file_model_weights = 'path_to_saved_model_weights.pt'  # Update this path
    dir_save_preds = 'path_to_save_predictions'  # Update this path



In [ ]:
# Call the batch_train function with the FLAGS object
import time

batch_train(FLAGS)


/content/resnet_34_deeplab_v3+/train_metrics.csv created successfully with header row
Dataset information:
Number of training samples: 162
Number of validation samples: 8

Training oil spill segmentation model: resnet_34_deeplab_v3+

----------------------------------------------------------------------------------------------------
Epoch : 1/50, Time: 2.38 sec., Train Loss: 0.90629
Validation Loss: 0.50072, Validation Accuracy: 0.81540, Validation IOU: 0.31417
----------------------------------------------------------------------------------------------------
Epoch : 2/50, Time: 2.75 sec., Train Loss: 0.72559
Validation Loss: 0.33680, Validation Accuracy: 0.86478, Validation IOU: 0.31788
----------------------------------------------------------------------------------------------------
Epoch : 3/50, Time: 2.69 sec., Train Loss: 0.58955
Validation Loss: 0.30454, Validation Accuracy: 0.93421, Validation IOU: 0.53641
----------------------------------------------------------------------

In [ ]:
import os
import numpy as np
import torch
from skimage.io import imsave
import torch.nn.functional as F


def create_directory(dir_path):
    """Create a directory if it doesn't exist."""
    if not os.path.isdir(dir_path):
        os.makedirs(dir_path)
        print(f"Created directory: {dir_path}")

from skimage.io import imsave

def inference_loop(
    dataset_loader, model, dir_labels, dir_masks, num_classes, device, image_format=".png", selected_images=None
):
    model.eval()
    infer_acc = 0
    infer_class_IOU = []

    dict_label_to_color_mapping = {
        0: np.array([0, 0, 0]),
        1: np.array([0, 255, 255]),
        2: np.array([255, 0, 0]),
        3: np.array([153, 76, 0]),
        4: np.array([0, 153, 0]),
    }

    for idx, (images, labels) in enumerate(dataset_loader):
        images = images.to(device, dtype=torch.float)
        labels = labels.to(device, dtype=torch.long)

        with torch.no_grad():
            pred_logits = model(images)
            pred_probs = F.softmax(pred_logits, dim=1)
            pred_label = torch.argmax(pred_probs, dim=1)

        infer_acc += compute_mean_pixel_acc(labels, pred_label)
        infer_class_IOU_cur_sample = compute_class_IOU(labels, pred_label)
        infer_class_IOU.append(infer_class_IOU_cur_sample)

        # Convert predictions to numpy arrays
        pred_label_arr = pred_label.cpu().numpy().squeeze()
        pred_label_one_hot = np.eye(num_classes)[pred_label_arr]

        pred_mask_arr = np.zeros((pred_label_arr.shape[0], pred_label_arr.shape[1], 3))
        for sem_class in range(num_classes):
            curr_class_label = pred_label_one_hot[:, :, sem_class]
            curr_class_color_mapping = dict_label_to_color_mapping[sem_class]
            pred_mask_arr += curr_class_label[..., None] * curr_class_color_mapping

        pred_label_arr = pred_label_arr.astype(np.uint8)
        pred_mask_arr = pred_mask_arr.astype(np.uint8)

        # Use the original image filename for saving the predictions
        if selected_images is not None:
            image_filename = selected_images[idx]
            base_filename = os.path.splitext(image_filename)[0]
        else:
            base_filename = f"pred_{idx}"

        # Save predictions
        file_pred_label = os.path.join(dir_labels, f"{base_filename}_label{image_format}")
        file_pred_mask = os.path.join(dir_masks, f"{base_filename}_mask{image_format}")

        imsave(file_pred_label, pred_label_arr)
        imsave(file_pred_mask, pred_mask_arr)

    infer_acc /= len(dataset_loader)
    infer_class_IOU = np.array(infer_class_IOU)
    infer_per_class_IOU = np.nanmean(infer_class_IOU, axis=0)
    return infer_acc, infer_per_class_IOU
def run_inference():
    """Run inference on a subset of 10 random images."""
    # Define the images and labels directories
    images_dir = dir_dataset  # This should point to the images directory
    labels_dir = dir_labels_dataset  # This should point to the labels directory

    # Create the dataset
    dataset = M4DSAROilSpillDataset(
        images_dir=images_dir,
        labels_dir=labels_dir,
        transform=image_transform,
    )

    # Get the list of image filenames
    list_inference_images = dataset.image_files

    # Select 10 random indices
    indices = np.random.choice(len(list_inference_images), 10, replace=False)

    # Create a subset of the dataset
    subset_dataset = torch.utils.data.Subset(dataset, indices)

    # Create a DataLoader for the subset
    subset_loader = torch.utils.data.DataLoader(subset_dataset, batch_size=1, shuffle=False)

    selected_images = [list_inference_images[i] for i in indices]

    print("Selected 10 random images for inference.")
    print(f"Number of test samples: {len(selected_images)}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if which_model == "resnet_34_deeplab_v3+":
        oil_spill_seg_model = ResNet34DeepLabV3Plus(num_classes=num_classes, pretrained=bool(pretrained))
    else:
        print("Model not implemented.")
        return

    oil_spill_seg_model.to(device)
    oil_spill_seg_model.load_state_dict(torch.load(file_model_weights, map_location=device))

    create_directory(dir_labels)
    create_directory(dir_masks)

    infer_acc, infer_per_class_IOU = inference_loop(
        subset_loader,
        oil_spill_seg_model,
        dir_labels,
        dir_masks,
        num_classes,
        device,
        image_format=".png",
        selected_images=selected_images,
    )

    infer_acc *= 100
    infer_per_class_IOU *= 100
    infer_IOU = np.mean(infer_per_class_IOU)

    print("Inference test set metrics")
    print(f"Accuracy: {infer_acc:.3f}%")
    print(f"Mean IOU: {infer_IOU:.3f}%")
    print("Per class IOU:")
    print(infer_per_class_IOU)


In [ ]:
# Define the parameters directly in the notebook
dir_dataset = "/content/drive/Othercomputers/MiPC/filtered_patches/test/images"  # Update as needed
dir_labels_dataset = "/content/drive/Othercomputers/MiPC/filtered_patches/test/labels_1D"
num_classes = 5
which_model = "resnet_34_deeplab_v3+"
file_model_weights = "/content/resnet_34_deeplab_v3+/oil_spill_seg_resnet_34_deeplab_v3+_17.pt"
dir_save_preds = "./predictions/"
dir_labels = os.path.join(dir_save_preds, "labels")
dir_masks = os.path.join(dir_save_preds, "masks")
pretrained = 1

In [ ]:
run_inference()


Selected 10 random images for inference.
Number of test samples: 10


<ipython-input-77-9c3bf42d64ff>:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  oil_spill_seg_model.load_state_dict(torch.load(file_model_weights, map_location=device))
<

Inference test set metrics
Accuracy: 90.210%
Mean IOU: nan%
Per class IOU:
[90.7203385  49.25614243  0.                 nan 76.79179858]


<ipython-input-77-9c3bf42d64ff>:68: UserWarning: ./predictions/labels/img_0041_23_label.png is a low contrast image
  imsave(file_pred_label, pred_label_arr)
<ipython-input-77-9c3bf42d64ff>:73: RuntimeWarning: Mean of empty slice
  infer_per_class_IOU = np.nanmean(infer_class_IOU, axis=0)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
inference_dataset_loader, list_inference_images = get_dataloader_for_inference(dir_dataset)
print(list_inference_images)  # Check if the image paths are correct